# Resonator quickstart: two nodes in one notebook

This notebook creates two resonator nodes in one kernel, admits them to
each other, and walks the day-1 surface: SQL and SPARQL round trips over
the wire, the local owner channel, and chat.

Install the `resonator` package first (a prebuilt wheel, or
`maturin build --release` in `crates/python` and `pip install` the wheel).
`pandas` is optional but used below.

A node is one directory holding its sqlite database (`rsntr.db`) and its
ed25519 key (`rsntr.key`); constructing a `Node` initializes the directory
on first use. `offline=True` binds to localhost with no relays: right for
same-machine demos like this one. Leave it off to reach peers across
networks via the n0 relays (two Colab VMs meet that way; see the note at
the end).

In [1]:
import os, tempfile, time
import resonator

base = tempfile.mkdtemp(prefix="rsntr-quickstart-")
alice = resonator.Node(os.path.join(base, "alice"), offline=True)
bob = resonator.Node(os.path.join(base, "bob"), offline=True)
print(resonator.__version__)
print("alice", alice.endpoint_id)
print("bob  ", bob.endpoint_id)

0.1.0
alice 03b9de61f57fd7e699415f37cb4017806298e0a7c48e7ad7450934db2cf6475a
bob   311b37349fad279400f9d4f5d93a80eb851067d0eb2e56833fed611c01593b76


## Serve and admit

`serve()` starts the node in the background (iroh endpoint, serving
pipeline, outbox worker, presence). A ticket names the live endpoint and
pastes into the other node's `add_peer`, which admits that peer into
`_peers`. Admission is the gate: only admitted peers get past hello.

In [2]:
alice.serve()
bob.serve()

alice.add_peer("bob", bob.ticket())
bob.add_peer("alice", alice.ticket())
print(alice)
print(bob)

Node(dir="/var/folders/bv/qq7svdhx3n54_cjj2pxz1_1h0000gn/T/rsntr-quickstart-j1uwxy_i/alice", offline=true, serving=true)
Node(dir="/var/folders/bv/qq7svdhx3n54_cjj2pxz1_1h0000gn/T/rsntr-quickstart-j1uwxy_i/bob", offline=true, serving=true)


## SQL over the wire

Bob writes his own tables over the owner channel (`local()` skips the
peer gate and the authenticator chain but stays footprint-collected and
audited; DDL is permitted there). What alice may do is decided by bob's
`_policy` table: one allow row for `read` on `notes`.

In [3]:
bob.local("CREATE TABLE notes (id INTEGER PRIMARY KEY, body TEXT)")
bob.local("INSERT INTO notes (body) VALUES (?), (?)",
          params=["first note", "second note"])
bob.local("INSERT INTO _policy (peer_or_group, table_name, action, effect, note) "
          "VALUES (?, 'notes', 'read', 'allow', 'quickstart')",
          params=[alice.endpoint_id])

res = alice.query("bob", "SELECT id, body FROM notes WHERE id > ?", params=[0])
print(res.columns, res.rows)
res.to_pandas()

['id', 'body'] [[1, 'first note'], [2, 'second note']]


,id,body
0,1,first note
1,2,second note


Anything the policy does not allow raises `resonator.Denied`:

In [4]:
try:
    alice.query("bob", "SELECT * FROM _peers")
except resonator.Denied as e:
    print("denied:", e)

denied: no decider in the chain allowed the request


## SPARQL

Every node carries the RDF store (`rdf_terms`/`rdf_triples`).
`load_turtle()` loads Turtle locally; `sparql()` queries the own store
over the owner channel. Remotely the `sparql` modulation rides the same
envelopes as SQL, gated by policy on the store's backing tables;
CONSTRUCT answers as Turtle text.

In [5]:
n = bob.load_turtle("""
@prefix ex: <http://example.org/> .
ex:alice ex:knows ex:bob ; ex:name \"Alice\" .
ex:bob ex:name \"Bob\" .
""")
print("loaded", n, "triples")

for table in ("rdf_triples", "rdf_terms"):
    bob.local("INSERT INTO _policy (peer_or_group, table_name, action, effect, note) "
              "VALUES (?, ?, 'read', 'allow', 'quickstart')",
              params=[alice.endpoint_id, table])

r = alice.query("bob", "SELECT ?p ?o WHERE { <http://example.org/alice> ?p ?o }",
                mod="sparql")
print(r.to_dicts())
print(alice.query("bob", "CONSTRUCT { ?s ?p ?o } WHERE { ?s ?p ?o }", mod="sparql"))

bob.sparql("SELECT ?s ?p ?o WHERE { ?s ?p ?o }").to_pandas()

loaded 3 triples
[{'p': '<http://example.org/knows>', 'o': '<http://example.org/bob>'}, {'p': '<http://example.org/name>', 'o': '"Alice"'}]
<http://example.org/alice> <http://example.org/knows> <http://example.org/bob> .
<http://example.org/alice> <http://example.org/name> "Alice" .
<http://example.org/bob> <http://example.org/name> "Bob" .



,s,p,o
0,<http://example.org/alice>,<http://example.org/knows>,<http://example.org/bob>
1,<http://example.org/alice>,<http://example.org/name>,"""Alice"""
2,<http://example.org/bob>,<http://example.org/name>,"""Bob"""


## Chat

`chat_init()` scaffolds the chat tables, projection points, and policy
(direct messages open to admitted peers). A send appends locally and
enqueues the delivery in `_outbox`; the serving node's outbox worker
carries it over. `chat_log()` is a local read, newest first; outgoing
rows carry their delivery status.

In [6]:
alice.chat_init()
bob.chat_init()

sent = alice.chat_send("bob", "hello bob, from a notebook")
print("queued", sent["message_id"], "to", sent["queued_to"][0][:12], "...")
alice.wake_outbox()

deadline = time.time() + 20
while time.time() < deadline and not bob.chat_log("alice"):
    time.sleep(0.3)
print("bob sees:", [m["body"] for m in bob.chat_log("alice")])

bob.chat_send("alice", "hi alice, got it")
bob.wake_outbox()
while time.time() < deadline and not [m for m in alice.chat_log("bob") if not m["outgoing"]]:
    time.sleep(0.3)
for m in alice.chat_log("bob"):
    print(("->" if m["outgoing"] else "<-"), m["body"], "", m["status"] or "")

queued 01KYS2MCGHPWX9X2S5QDTT0FDW to 311b37349fad ...


bob sees: ['hello bob, from a notebook']


<- hi alice, got it  
-> hello bob, from a notebook  done


## Help, then hang up

`help()` asks a peer for usage guidance (the one modulation every node
must serve, even to strangers). `stop()` shuts the node down; `Node` is
also a context manager, so `with resonator.Node(...) as node:` stops on
exit. The database stays plain WAL sqlite: open `node.db_path` with the
stock `sqlite3` module or pandas any time, even while serving.

In [7]:
print(alice.help("bob"))

This is a resonator node.
Send an rsntr:Query with modulation 'sql-sqlite' to run SQL, 'sparql' to query the RDF store, 'projection' for my capability menu, or 'help' for usage.

This is a resonator node (rsntr, envelope 0.1).
I serve the modulations: help, sql-sqlite, sparql, projection, media, chat.
Readable now: notes, rdf_triples, rdf_terms.
Not admitted yet? Introduce yourself and I may let you in:
  [] a rsntr:Knock ; rsntr:message "who you are and what you want" .
Ask for more: send a help query with text one of: modulations, tables, knock, projection, examples.


In [8]:
alice.stop()
bob.stop()
print(alice.is_serving, bob.is_serving)

False False


## Across machines (Colab to Colab)

Drop `offline=True` and the same flow works between two notebooks on
different machines: each runs `serve()`, they exchange `ticket()` strings
(which then include relay routing), `add_peer` each other, and query.
Verifying that relay leg notebook-to-notebook is a manual, networked
follow-up; this notebook stays offline so it executes anywhere.